# Production Hybrid Retrieval with Databricks Vector Search

Demonstrating production-ready **Hybrid Retrieval** using Databricks Vector Search with `hr_rag_index_200`.

**Key Features:**
* 🔀 **Hybrid Search**: Automatic BM25 + Vector fusion
* 🎯 **Simple API**: Just pass `query_text` - no manual embeddings
* ✅ **Quality Filtering**: Score thresholds for confidence
* 🚀 **Production Ready**: Built-in Databricks integration

**Best Practices:**
* Hybrid search outperforms pure semantic on acronyms (DSAR, MFA, eNPS)
* Set score thresholds to reject low-confidence results
* Hybrid = keyword precision + semantic understanding

In [0]:
# PRODUCTION-READY HYBRID SEARCH
# Using hr_rag_index_200 with embedding endpoint for true hybrid search
# Includes score thresholds for quality filtering

from databricks.sdk import WorkspaceClient
import pandas as pd
import json

w = WorkspaceClient()

print(" PRODUCTION-READY HYBRID SEARCH")
print("=" * 100)
print("Approach: query_text + hybrid search + score threshold")
print("Index: hr_rag_index_200 (with databricks-gte-large-en endpoint)\n")

query = "What is DSAR?"
MIN_SCORE = 0.50  # Quality threshold for results

try:
    print(f"Query: {query}")
    print(f"Score threshold: {MIN_SCORE}\n")
    print("Executing hybrid search...\n")
    
    # Query hr_rag_index_200 with hybrid search
    results = w.vector_search_indexes.query_index(
        index_name="hr_catalog.hr_core.hr_rag_index_200",
        query_text=query,
        columns=["id", "content", "metadata"],  # Columns from hr_document_chunks_200
        query_type="HYBRID",  # BM25 + Vector
        score_threshold=MIN_SCORE,
        num_results=10
    )
    
    if results.result and results.result.data_array:
        print("=" * 100)
        print(f" HIGH-CONFIDENCE RESULTS (score >= {MIN_SCORE})")
        print("=" * 100)
        print(f"\nFound {len(results.result.data_array)} results\n")
        
        result_data = []
        for i, row in enumerate(results.result.data_array, 1):
            chunk_id = row[0]
            content = row[1]
            metadata_json = row[2]
            score = row[-1]
            
            # Parse metadata JSON
            try:
                metadata = json.loads(metadata_json)
                document_name = metadata.get('document_name', 'Unknown')
            except:
                document_name = 'Unknown'
            
            print(f"\n{i}. Hybrid Score: {score:.4f}")
            print(f"   Document: {document_name}")
            print(f"   Content: {content[:300]}...")
            print("-" * 100)
            
            result_data.append({
                "Rank": i,
                "Score": round(score, 4),
                "Document": document_name[:50]
            })
        
        print("\n" + "=" * 100)
        print("SUMMARY")
        print("=" * 100)
        df = pd.DataFrame(result_data)
        display(df)
        
        print("\n" + "=" * 100)
        print("PRODUCTION SUCCESS")
        print("=" * 100)
        print(f"""
Found {len(results.result.data_array)} high-confidence results
Hybrid search (BM25 + Vector) finds relevant content
Built-in score threshold ensured quality
Simple query_text API - no manual embeddings needed
Using: hr_catalog.hr_core.hr_rag_index_200
        """)
    else:
        print("=" * 100)
        print(f" NO HIGH-CONFIDENCE RESULTS (all scores < {MIN_SCORE})")
        print("=" * 100)
        print("\nNo results met the quality threshold.")
        print("\nProduction response: 'I don't have information about that topic.'")
        
except Exception as e:
    print(f"\n Error: {str(e)}")
    import traceback
    traceback.print_exc()

 PRODUCTION-READY HYBRID SEARCH
Approach: query_text + hybrid search + score threshold
Index: hr_rag_index_200 (with databricks-gte-large-en endpoint)

Query: What is DSAR?
Score threshold: 0.5

Executing hybrid search...

 HIGH-CONFIDENCE RESULTS (score >= 0.5)

Found 1 results


1. Hybrid Score: 1.0000
   Document: GDPR_Employee_Data_Practices_2025-01.pdf
   Content: GDPR Employee Data Practices 2025-01

Owner: noah.nowak@nordstar.example.com (HR)
Version: 2025-01
• Lawful bases overview
• DSAR intake ® response
• Retention map snapshot
• Vendors & DPAs
• Cross-bo...
----------------------------------------------------------------------------------------------------

SUMMARY


Rank,Score,Document
1,1.0,GDPR_Employee_Data_Practices_2025-01.pdf



PRODUCTION SUCCESS

Found 1 high-confidence results
Hybrid search (BM25 + Vector) finds relevant content
Built-in score threshold ensured quality
Simple query_text API - no manual embeddings needed
Using: hr_catalog.hr_core.hr_rag_index_200
        


In [0]:
# DEMONSTRATION: When Hybrid Search Outperforms Pure Semantic Search
# Testing with acronym-heavy and specific-term queries

from databricks.sdk import WorkspaceClient
import pandas as pd
import json

w = WorkspaceClient()

print(" COMPARING SEMANTIC vs HYBRID SEARCH")
print("=" * 100)
print("Hypothesis: Hybrid search outperforms pure semantic on:")
print("  • Acronym queries (DSAR, GDPR, etc.)")
print("  • Specific technical terms")
print("  • Short queries with exact keywords\n")

# Test queries that favor hybrid over pure semantic
test_queries = [
    "What is DSAR?",  # Acronym query
    "MFA fatigue",     # Technical term
    "eNPS score"       # Specific metric acronym
]

results_comparison = []

for query in test_queries:
    print(f"\n{'=' * 100}")
    print(f"Query: '{query}'")
    print("=" * 100)
    
    # Test 1: HYBRID search
    try:
        hybrid_results = w.vector_search_indexes.query_index(
            index_name="hr_catalog.hr_core.hr_rag_index_200",
            query_text=query,
            columns=["id", "content", "metadata"],
            query_type="HYBRID",
            num_results=1
        )
        
        if hybrid_results.result and hybrid_results.result.data_array:
            hybrid_score = hybrid_results.result.data_array[0][-1]
            hybrid_content = hybrid_results.result.data_array[0][1][:100]
            
            try:
                metadata = json.loads(hybrid_results.result.data_array[0][2])
                hybrid_doc = metadata.get('document_name', 'Unknown')
            except:
                hybrid_doc = 'Unknown'
        else:
            hybrid_score = 0.0
            hybrid_doc = 'No results'
            hybrid_content = 'N/A'
    except Exception as e:
        hybrid_score = 0.0
        hybrid_doc = f'Error: {str(e)[:50]}'
        hybrid_content = 'N/A'
    
    # Test 2: VECTOR-only search (semantic)
    try:
        vector_results = w.vector_search_indexes.query_index(
            index_name="hr_catalog.hr_core.hr_rag_index_200",
            query_text=query,
            columns=["id", "content", "metadata"],
            query_type="ANN",  # Pure vector/semantic search
            num_results=1
        )
        
        if vector_results.result and vector_results.result.data_array:
            vector_score = vector_results.result.data_array[0][-1]
            vector_content = vector_results.result.data_array[0][1][:100]
            
            try:
                metadata = json.loads(vector_results.result.data_array[0][2])
                vector_doc = metadata.get('document_name', 'Unknown')
            except:
                vector_doc = 'Unknown'
        else:
            vector_score = 0.0
            vector_doc = 'No results'
            vector_content = 'N/A'
    except Exception as e:
        vector_score = 0.0
        vector_doc = f'Error: {str(e)[:50]}'
        vector_content = 'N/A'
    
    # Compare scores
    improvement = ((hybrid_score - vector_score) / vector_score * 100) if vector_score > 0 else 0
    winner = "Hybrid" if hybrid_score > vector_score else ("Semantic" if vector_score > hybrid_score else "Tie")
    
    print(f"\n Results:")
    print(f"  Hybrid Score:   {hybrid_score:.4f} → {hybrid_doc[:40]}")
    print(f"  Semantic Score: {vector_score:.4f} → {vector_doc[:40]}")
    print(f"  Winner: {winner}")
    if improvement > 0:
        print(f"  Improvement: +{improvement:.1f}%")
    
    results_comparison.append({
        "Query": query,
        "Hybrid Score": round(hybrid_score, 4),
        "Semantic Score": round(vector_score, 4),
        "Improvement %": round(improvement, 1) if improvement > 0 else 0,
        "Winner": winner
    })

print("\n" + "=" * 100)
print("OVERALL COMPARISON")
print("=" * 100)
df = pd.DataFrame(results_comparison)
display(df)


 COMPARING SEMANTIC vs HYBRID SEARCH
Hypothesis: Hybrid search outperforms pure semantic on:
  • Acronym queries (DSAR, GDPR, etc.)
  • Specific technical terms
  • Short queries with exact keywords


Query: 'What is DSAR?'

 Results:
  Hybrid Score:   1.0000 → GDPR_Employee_Data_Practices_2025-01.pdf
  Semantic Score: 0.5219 → GDPR_Employee_Data_Practices_2025-01.pdf
  Winner: Hybrid
  Improvement: +91.6%

Query: 'MFA fatigue'

 Results:
  Hybrid Score:   0.9692 → Security_Awareness_Brief_2025-01.pdf
  Semantic Score: 0.5254 → HR_Quarterly_Summary_2025Q1.pdf
  Winner: Hybrid
  Improvement: +84.5%

Query: 'eNPS score'

 Results:
  Hybrid Score:   1.0000 → HR_Quarterly_Summary_2025Q1.pdf
  Semantic Score: 0.5867 → HR_Quarterly_Summary_2025Q1.pdf
  Winner: Hybrid
  Improvement: +70.4%

OVERALL COMPARISON


Query,Hybrid Score,Semantic Score,Improvement %,Winner
What is DSAR?,1.0,0.5219,91.6,Hybrid
MFA fatigue,0.9692,0.5254,84.5,Hybrid
eNPS score,1.0,0.5867,70.4,Hybrid
